# Gloria — Bone Fracture Model Training (Google Colab)

**Before you start:** Runtime → **Change runtime type** → **T4 GPU** (or better).

This notebook clones the project, loads training data, trains the MobileNetV2 multi-task model, evaluates on the test set, and exports a TFLite file for the Flutter app.

## 1. Get the project code

Your GitHub repo is **private**, so a plain `git clone` fails in Colab. Use **one** option below.

| Option | Steps |
|--------|--------|
| **A — Make repo public** (easiest) | GitHub → repo **Settings** → **Danger Zone** → **Change visibility** → Public. Then run the clone cell. |
| **B — GitHub token** (keep private) | Create a token at https://github.com/settings/tokens (scope: `repo`). Paste it in the token cell below. |
| **C — Code zip on Drive** (no Git) | Zip your local `ml` folder → upload `gloria_ml.zip` to Drive → run the Drive cell below. |

In [ ]:
# --- Option A or B: clone from GitHub ---
USE_GITHUB_TOKEN = True   # set False only if repo is PUBLIC

if USE_GITHUB_TOKEN:
    from getpass import getpass
    token = getpass("Paste GitHub token (repo scope): ")
    REPO_URL = f"https://{token}@github.com/CaL7598/offline-bone-fracture-detection-app.git"
else:
    REPO_URL = "https://github.com/CaL7598/offline-bone-fracture-detection-app.git"

!git clone --depth 1 "{REPO_URL}" gloria
%cd gloria/ml

### Option C — Load code from Google Drive (skip git entirely)

1. On your PC: zip the folder `c:\Users\boaky\Music\gloria\ml` → name it `gloria_ml.zip`
2. Upload to Google Drive (e.g. `MyDrive/gloria/gloria_ml.zip`)
3. Set `USE_DRIVE_CODE = True` and run:

In [ ]:
USE_DRIVE_CODE = False  # set True if you uploaded gloria_ml.zip
CODE_ZIP = "/content/drive/MyDrive/gloria/gloria_ml.zip"

if USE_DRIVE_CODE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/gloria
    !unzip -q -o "{CODE_ZIP}" -d /content/gloria/
    %cd /content/gloria/ml
    import os
    print("Ready at", os.getcwd())

## 2. Install dependencies

Colab already includes TensorFlow with GPU support.

In [ ]:
!pip install -q -r requirements.txt

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 3. Get training data

Pick **one** option below.

| Option | When to use |
|--------|-------------|
| **A — Repo data** | Clone already includes `ml/data/raw/` (train/valid/test) |
| **B — Google Drive zip** | You zipped `ml/data/raw` locally and uploaded it to Drive |
| **C — Prepare from MURA/FracAtlas** | You uploaded dataset zips to Drive and want to run `prepare_datasets.py` |

### Option A — Use data from the cloned repo (skip if empty)

In [ ]:
from pathlib import Path

RAW = Path("data/raw")
exts = {".jpg", ".jpeg", ".png", ".bmp"}

def count_split(name):
    d = RAW / name
    if not d.exists():
        return 0
    return sum(1 for f in d.rglob("*") if f.suffix.lower() in exts)

for split in ["train", "valid", "test"]:
    print(f"{split}: {count_split(split)} images")

if count_split("valid") == 0:
    print("\n⚠️  Validation folder is empty. Use Option B or C, or re-run prepare_datasets.py.")

### Option B — Restore `data/raw` from Google Drive

1. On your PC, zip the folder `ml/data/raw` → `gloria_raw.zip`
2. Upload to Google Drive (e.g. `MyDrive/gloria/gloria_raw.zip`)
3. Run the cell below (edit `ZIP_PATH` if needed)

In [ ]:
USE_DRIVE_ZIP = False  # set True after uploading your zip
ZIP_PATH = "/content/drive/MyDrive/gloria/gloria_raw.zip"

if USE_DRIVE_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    !unzip -q -o "{ZIP_PATH}" -d data/
    print("Extracted to data/raw/")

### Option C — Prepare from MURA + FracAtlas on Drive

Upload unzipped MURA and FracAtlas under Drive, then run:

In [ ]:
PREPARE_FROM_DOWNLOADS = False  # set True when downloads are on Drive

if PREPARE_FROM_DOWNLOADS:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p data/downloads/mura data/downloads/fracatlas
    # Edit these paths to match your Drive layout:
    !cp -r "/content/drive/MyDrive/gloria/mura/." data/downloads/mura/
    !cp -r "/content/drive/MyDrive/gloria/fracatlas/." data/downloads/fracatlas/
    !python data/prepare_datasets.py

## 4. Train the model

Default: 20 epochs, batch size 32, early stopping on validation fracture accuracy.

In [ ]:
!mkdir -p checkpoints export

EPOCHS = 20
BATCH_SIZE = 32

!python train.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --save checkpoints/bone_fracture_model.keras

## 5. Evaluate on the test set

In [ ]:
!python evaluate.py --model checkpoints/bone_fracture_model.keras

## 6. Export TensorFlow Lite (for the Flutter app)

In [ ]:
# Use --float32 if FP16 fails on your Android device
!python export_tflite.py --model checkpoints/bone_fracture_model.keras --output export/bone_fracture.tflite

## 7. Download artifacts to your PC

Copy these into your local project:
- `checkpoints/bone_fracture_model.keras` → `ml/checkpoints/`
- `export/bone_fracture.tflite` → `ml/export/` and `mobile/assets/`

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("/content/gloria_model_artifacts", "zip", ".", "checkpoints")
shutil.make_archive("/content/gloria_tflite", "zip", ".", "export")

files.download("/content/gloria_model_artifacts.zip")
files.download("/content/gloria_tflite.zip")

## 8. (Optional) Save to Google Drive instead of downloading

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_OUT = "/content/drive/MyDrive/gloria/training_output"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p "{DRIVE_OUT}"
    !cp -r checkpoints export "{DRIVE_OUT}/"
    print("Saved to", DRIVE_OUT)